# Phase 0 — Sélection du Modèle VLM / VLM Model Selection Study
**Projet / Project:** Système d'Inspection des Ouvrages d'Art par IA (CID)  
**Auteur / Author:** Youssef Elhardouzi  
**Date:** Avril 2026  
**Version:** v4 — Évaluation étendue 10 images + baseline zero-shot

---

## Méthodologie de sélection / Selection Methodology

Les candidats ont été sélectionnés en deux phases :

**Phase initiale** — Open VLM Leaderboard (OpenCompass, consultation Mars 2026), top 15, filtrés par :
1. Paramètres ≤ 4B (contrainte VRAM T4)
2. Disponibilité publique sur HuggingFace Hub
3. Interface standard AutoModel
4. Support de fine-tuning confirmé dans LLaMA-Factory
5. Licence ouverte pour usage commercial (Apache 2.0 ou MIT)

**Addendum Phase 0** — Révision systématique des modèles LLaMA-Factory face aux sorties récentes HuggingFace (Mars–Avril 2026) avec les mêmes critères.

**Modèles exclus avec justification :**
- `SmolVLM2-2.2B` — pas de template LLaMA-Factory
- `Gemma3-4B` — 4B à la limite VRAM T4 ; licence Gemma Terms of Use
- `Qwen2.5-Omni-3B` — licence qwen-research (recherche uniquement)
- `Qwen3.5-VL` — n'existe pas (Qwen3.5 est texte uniquement)
- `LFM2.5-VL` — licence LFM 1.0 (restreint aux entreprises < 10M$ CA)

**Candidats retenus :**

| Modèle | Params | Licence | LF template | Source |
|--------|--------|---------|-------------|--------|
| InternVL3-2B | 2B | MIT | `intern_vl` | Leaderboard rank #10 |
| Qwen2.5-VL-3B | 3B | Apache 2.0 | `qwen2_5_vl` | Leaderboard rank #11 |
| Qwen3-VL-2B | 2B | Apache 2.0 | `qwen3_vl` | Addendum (arXiv:2505.09388) |
| InternVL3.5-2B | 2.35B | Apache 2.0 | `intern_vl` | Addendum (arXiv:2508.18265) |

---
## 0 — Installation

Installe toutes les dépendances. À exécuter une seule fois.  
**Redémarrez le kernel manuellement après cette cellule.**

In [ ]:
import subprocess, sys

packages = [
    'transformers>=4.57.0',
    'accelerate>=0.29.0',
    'bitsandbytes>=0.43.0',
    'qwen-vl-utils',
    'pillow',
    'torchvision',
    'pandas',
    'rouge-score',
    'nltk',
    'python-dotenv',
]

for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

import nltk
nltk.download('punkt',     quiet=True)
nltk.download('punkt_tab', quiet=True)

print('✓ Installation terminée.')
print('IMPORTANT: Redémarrez le kernel maintenant avant de continuer.')

---
## 1 — Configuration

In [ ]:
import os, sys, json, gc, time, base64, textwrap, warnings, random
import pandas as pd
import torch
from io import BytesIO
from PIL import Image
from IPython.display import display
from dotenv import load_dotenv

warnings.filterwarnings('ignore')
load_dotenv()

# ── Mount Google Drive (Colab only) ───────────────────────────────────────
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    print('✓ Google Drive monté')
except ImportError:
    pass

# ─────────────────────────────────────────────────────────────────────────
# EDIT THESE if you don't use .env / Secrets
HF_TOKEN_OVERRIDE = ''
CID_BASE_OVERRIDE = ''   # e.g. '/content/drive/MyDrive/CID'
# ─────────────────────────────────────────────────────────────────────────

def _get_secret(key):
    overrides = {'HF_TOKEN': HF_TOKEN_OVERRIDE, 'CID_BASE': CID_BASE_OVERRIDE}
    if overrides.get(key): return overrides[key]
    if os.environ.get(key): return os.environ[key]
    try:
        from google.colab import userdata
        return userdata.get(key)
    except Exception: pass
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(key)
    except Exception: pass
    return None

# ── HuggingFace token ──────────────────────────────────────────────────────
HF_TOKEN = _get_secret('HF_TOKEN')
if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
    print('✓ HF_TOKEN : authentifié')
else:
    print('⚠ HF_TOKEN non trouvé — modèles publics uniquement')

# ── Resolve paths ─────────────────────────────────────────────────────────
CID_BASE = _get_secret('CID_BASE')
if not CID_BASE:
    notebook_dir = os.path.abspath('.')
    CID_BASE = os.path.dirname(notebook_dir) if os.path.basename(notebook_dir) in ('phase0', 'notebooks') else notebook_dir
    print(f'⚠ CID_BASE non défini — auto-détecté : {CID_BASE}')
else:
    print(f'✓ CID_BASE : {CID_BASE}')

JSON_DATA    = os.path.join(CID_BASE, 'data', 'finetuning_dataset.json')
RESULTS_DIR  = os.path.join(CID_BASE, 'phase0')
results_path = os.path.join(RESULTS_DIR, 'model_selection_results_v4.json')
os.makedirs(RESULTS_DIR, exist_ok=True)

# ── GPU check ──────────────────────────────────────────────────────────────
assert torch.cuda.is_available(), 'Aucun GPU détecté.'
print(f'✓ GPU  : {torch.cuda.get_device_name(0)}')
print(f'✓ VRAM : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
import transformers; print(f'✓ Transformers : {transformers.__version__}')

# ── Inspection prompt ──────────────────────────────────────────────────────
INSTRUCTION = (
    "Tu es un ingénieur civil spécialisé dans l'inspection des ouvrages d'art. "
    "Analyse cette image et rédige un commentaire d'inspection professionnel "
    "couvrant : le désordre observé, ses causes probables, sa gravité, "
    "et les suites à donner. Ne décris que ce qui est visuellement observable."
)

# ── ITSEOA structure keywords ──────────────────────────────────────────────
# Used only for qualitative reporting, not scoring
ITSEOA_KEYWORDS = {
    'cause':   ['cause', 'causé', 'origine', 'dû à', 'provoqué'],
    'gravité': ['gravité', 'grave', 'sévère', 'sévérité', 'criticité', 'urgent', 'critique'],
    'suites':  ['suite', 'recommand', 'préconise', 'intervention', 'traitement', 'réparation', 'surveill'],
}

# ─────────────────────────────────────────────────────────────────────────
# MODEL REGISTRY
# ─────────────────────────────────────────────────────────────────────────
MODEL_REGISTRY = [
    {
        'id':      'InternVL3-2B',
        'hf':      'OpenGVLab/InternVL3-2B-hf',
        'family':  'internvl',
        'lf_tpl':  'intern_vl',
        'license': 'MIT',
        'rank':    10,
        'avg':     64.5,
    },
    {
        'id':      'Qwen2.5-VL-3B',
        'hf':      'Qwen/Qwen2.5-VL-3B-Instruct',
        'family':  'qwen_vl',
        'lf_tpl':  'qwen2_5_vl',
        'license': 'Apache 2.0',
        'rank':    11,
        'avg':     64.5,
    },
    {
        'id':      'Qwen3-VL-2B',
        'hf':      'Qwen/Qwen3-VL-2B-Instruct',
        'family':  'qwen_vl',
        'lf_tpl':  'qwen3_vl',
        'license': 'Apache 2.0',
        'rank':    None,
        'avg':     None,
    },
    {
        'id':      'InternVL3.5-2B',
        'hf':      'OpenGVLab/InternVL3_5-2B-HF',
        'family':  'internvl',
        'lf_tpl':  'intern_vl',
        'license': 'Apache 2.0',
        'rank':    None,
        'avg':     None,
    },
]

print(f'\n✓ Configuration chargée')
print(f'  CID_BASE    : {CID_BASE}')
print(f'  JSON data   : {JSON_DATA}')
print(f'  Results dir : {RESULTS_DIR}')
print(f'  Modèles     : {[m["id"] for m in MODEL_REGISTRY]}')

---
## 2 — Chargement du Dataset et sélection des 10 images

Les 10 images sont sélectionnées de manière stratifiée par famille de désordre,
pour couvrir l'ensemble du spectre ITSEOA plutôt que de tirer aléatoirement.

In [ ]:
if not os.path.exists(JSON_DATA):
    raise FileNotFoundError(f'Dataset introuvable : {JSON_DATA}')

with open(JSON_DATA, encoding='utf-8') as f:
    records = json.load(f)
print(f'✓ Dataset : {len(records)} entrées')

# ── Stratified sample: 10 images across defect families ───────────────────
# Group records by family (fiche prefix = major defect category)
# Then pick ~2-3 per group to cover the full dataset range
# Fixed indices for reproducibility — covers biologique, béton armé,
# infiltration, maçonnerie, and métallique families
TEST_INDICES = [
    {'idx': 12,  'fiche': '2.9', 'family': 'Biologique / surface',     'desc': 'Désordres sur les parapets en maçonnerie'},
    {'idx': 25,  'fiche': '3.2', 'family': 'Biologique / surface',     'desc': 'Végétation sur tablier'},
    {'idx': 38,  'fiche': '4.2', 'family': 'Béton armé structural',    'desc': 'Éclats résultant de la corrosion des armatures'},
    {'idx': 52,  'fiche': '4.5', 'family': 'Béton armé structural',    'desc': 'Fissures de flexion'},
    {'idx': 67,  'fiche': '5.1', 'family': 'Maçonnerie',               'desc': 'Dégradation des joints'},
    {'idx': 80,  'fiche': '5.4', 'family': 'Maçonnerie',               'desc': 'Éclatement de pierres'},
    {'idx': 95,  'fiche': '6.3', 'family': 'Métallique',               'desc': 'Corrosion généralisée'},
    {'idx': 108, 'fiche': '6.7', 'family': 'Métallique',               'desc': 'Déformation de membrure'},
    {'idx': 116, 'fiche': '7.2', 'family': "Infiltration d'eau",       'desc': 'Écoulement en intrados de tablier'},
    {'idx': 130, 'fiche': '7.5', 'family': "Infiltration d'eau",       'desc': 'Stalactites et dépôts calcaires'},
]

# Clamp indices to dataset size in case dataset is smaller
TEST_INDICES = [t for t in TEST_INDICES if t['idx'] < len(records)]
if len(TEST_INDICES) < 10:
    print(f'⚠ Dataset contient {len(records)} entrées — seulement {len(TEST_INDICES)} indices valides')
    print('  Les indices manquants seront remplacés par des indices aléatoires.')
    existing = {t['idx'] for t in TEST_INDICES}
    remaining = [i for i in range(len(records)) if i not in existing]
    random.seed(42)
    extra = random.sample(remaining, min(10 - len(TEST_INDICES), len(remaining)))
    for idx in sorted(extra):
        r = records[idx]
        meta = r.get('meta', {})
        TEST_INDICES.append({
            'idx': idx,
            'fiche': meta.get('fiche', '?'),
            'family': meta.get('family', '?'),
            'desc': meta.get('desc', '?'),
        })
    TEST_INDICES = sorted(TEST_INDICES, key=lambda x: x['idx'])

# ── Load images and references ─────────────────────────────────────────────
test_samples = []
for t in TEST_INDICES:
    r          = records[t['idx']]
    img        = Image.open(BytesIO(base64.b64decode(r.get('image_b64', '')))).convert('RGB')
    commentary = r.get('commentary_full') or r.get('response', '')
    test_samples.append({
        'idx':       t['idx'],
        'fiche':     t['fiche'],
        'family':    t['family'],
        'desc':      t['desc'],
        'image':     img,
        'commentary': commentary,
    })
    print(f'  [{t["idx"]:>3}] Fiche {t["fiche"]} | {t["family"]} — {t["desc"]}')

print(f'\n✓ {len(test_samples)} images chargées')

---
## 3 — ModelWrapper : interface unifiée

In [ ]:
from transformers import (
    BitsAndBytesConfig, AutoProcessor,
    AutoModelForImageTextToText,
)

BNB_4BIT = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)


def free_gpu():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()


def vram_used_gb():
    return torch.cuda.memory_allocated() / 1e9


class ModelWrapper:
    """
    Unified loader and runner for all VLM families.
    Supported families:
      'internvl' — InternVL3 (HF format) and InternVL3.5 (HF format)
      'qwen_vl'  — Qwen2.5-VL and Qwen3-VL
    """

    def __init__(self, entry: dict):
        self.id     = entry['id']
        self.hf     = entry['hf']
        self.family = entry['family']
        self.model  = None
        self.proc   = None

    def load(self):
        self.proc = AutoProcessor.from_pretrained(self.hf, trust_remote_code=True)

        if self.family == 'internvl':
            # Both InternVL3-hf and InternVL3.5-HF use AutoModelForImageTextToText
            self.model = AutoModelForImageTextToText.from_pretrained(
                self.hf,
                quantization_config=BNB_4BIT,
                device_map='auto',
                trust_remote_code=True,
            )

        elif self.family == 'qwen_vl':
            # AutoModelForImageTextToText handles both Qwen2.5-VL and Qwen3-VL
            self.model = AutoModelForImageTextToText.from_pretrained(
                self.hf,
                quantization_config=BNB_4BIT,
                device_map='auto',
                trust_remote_code=True,
            )

        else:
            raise ValueError(f'Unknown family: {self.family!r}')

        self.model.eval()

    def generate(self, image: Image.Image, prompt: str, max_new_tokens: int = 400) -> str:
        inputs = self._format_inputs(image, prompt)
        with torch.no_grad():
            ids = self.model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
        n_input = inputs['input_ids'].shape[1]
        return self.proc.decode(ids[0][n_input:], skip_special_tokens=True)

    def _format_inputs(self, image: Image.Image, prompt: str) -> dict:
        messages = [{
            'role': 'user',
            'content': [{'type': 'image', 'image': image}, {'type': 'text', 'text': prompt}],
        }]

        if self.family == 'internvl':
            text   = self.proc.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
            inputs = self.proc(text=[text], images=[image], return_tensors='pt')
            return {k: v.to(self.model.device) for k, v in inputs.items()}

        elif self.family == 'qwen_vl':
            from qwen_vl_utils import process_vision_info
            text       = self.proc.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            imgs, vids = process_vision_info(messages)
            inputs     = self.proc(
                text=[text], images=imgs, videos=vids,
                padding=True, return_tensors='pt',
            )
            return {k: v.to(self.model.device) for k, v in inputs.items()}

    def unload(self):
        try:
            del self.model
            del self.proc
        except Exception:
            pass
        self.model = None
        self.proc  = None
        free_gpu()


print('✓ ModelWrapper initialisé')

---
## 4 — Boucle d'évaluation / Evaluation Loop

Résultats sauvegardés après chaque modèle. Reprise automatique si interruption.

> ⚠ **Durée estimée : 90–120 minutes** sur T4 Colab gratuit (4 modèles × 10 images).

In [ ]:
if os.path.exists(results_path):
    with open(results_path) as f:
        results = json.load(f)
    done_ids = {r['model_id'] for r in results if r['status'] == 'success'}
    print(f'✓ Résultats existants : {len(results)} entrées')
    print(f'  Déjà évalués avec succès : {done_ids or "aucun"}')
else:
    results, done_ids = [], set()

for entry in MODEL_REGISTRY:
    mid = entry['id']
    if mid in done_ids:
        print(f'\n  ↩ {mid} — déjà évalué, ignoré'); continue

    print(f'\n{"="*65}')
    print(f'  Évaluation : {mid}')
    print(f'  HF         : {entry["hf"]}')
    print(f'{"="*65}')

    result = {
        'model_id':         mid,
        'hf':               entry['hf'],
        'family':           entry['family'],
        'lf_template':      entry['lf_tpl'],
        'license':          entry['license'],
        'leaderboard_rank': entry['rank'],
        'leaderboard_avg':  entry['avg'],
        'vram_gb':          None,
        'avg_time_s':       None,
        'avg_tok_per_s':    None,
        'outputs':          [],
        'error':            None,
        'status':           'failed',
    }

    wrapper = ModelWrapper(entry)

    try:
        free_gpu()
        vram_before = vram_used_gb()
        print(f'  Chargement (4-bit NF4)...')
        t0 = time.time()
        wrapper.load()
        result['vram_gb'] = round(vram_used_gb() - vram_before, 2)
        print(f'  ✓ Chargé en {time.time()-t0:.0f}s · VRAM : {result["vram_gb"]} GB')

        times, tok_rates = [], []
        for s in test_samples:
            print(f'  ── [{s["idx"]:>3}] {s["family"]}')
            t0       = time.time()
            out      = wrapper.generate(s['image'], INSTRUCTION)
            elapsed  = round(time.time() - t0, 1)
            tok_rate = round(len(out.split()) / max(elapsed, 0.1), 1)
            times.append(elapsed); tok_rates.append(tok_rate)

            # ITSEOA keyword detection (qualitative only)
            out_lower = out.lower()
            itseoa_hits = {
                section: any(kw in out_lower for kw in kws)
                for section, kws in ITSEOA_KEYWORDS.items()
            }

            result['outputs'].append({
                'idx':         s['idx'],
                'fiche':       s['fiche'],
                'family':      s['family'],
                'desc':        s['desc'],
                'output':      out,
                'reference':   s['commentary'],
                'time_s':      elapsed,
                'tok_per_s':   tok_rate,
                'itseoa_hits': itseoa_hits,
            })
            hit_str = ' '.join(f'{k}={"✓" if v else "✗"}' for k, v in itseoa_hits.items())
            print(f'     {elapsed}s · {tok_rate} tok/s · {hit_str}')

        result['avg_time_s']    = round(sum(times) / len(times), 1)
        result['avg_tok_per_s'] = round(sum(tok_rates) / len(tok_rates), 1)
        result['status']        = 'success'
        print(f'\n  ✓ Temps moyen : {result["avg_time_s"]}s · {result["avg_tok_per_s"]} tok/s')

    except Exception as e:
        result['error'] = str(e)
        print(f'  ✗ ÉCHEC : {e}')
    finally:
        wrapper.unload()

    results = [r for r in results if r['model_id'] != mid] + [result]
    with open(results_path, 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    print(f'  ✓ Sauvegardé : {results_path}')

n_ok = sum(1 for r in results if r['status'] == 'success')
print(f'\n{"="*65}')
print(f'ÉVALUATION TERMINÉE — {n_ok}/{len(MODEL_REGISTRY)} modèles réussis')

---
## 5 — Scores BLEU & ROUGE-L + Baseline zero-shot

La **baseline zero-shot** (sortie vide) représente le score plancher — ROUGE-L = 0.  
Tous les modèles doivent dépasser ce plancher pour être pertinents.

In [ ]:
try:
    assert results
except Exception:
    with open(results_path) as f:
        results = json.load(f)
    print(f'✓ Rechargé ({len(results)} modèles)')

from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer as rs_module

smoother = SmoothingFunction().method1
r_scorer  = rs_module.RougeScorer(['rougeL'], use_stemmer=False)


def bleu(ref, hyp):
    if not hyp.strip(): return 0.0
    return round(sentence_bleu(
        [ref.lower().split()], hyp.lower().split(),
        weights=(0.5, 0.5), smoothing_function=smoother,
    ), 4)


def rouge_l(ref, hyp):
    if not hyp.strip(): return 0.0
    return round(r_scorer.score(ref, hyp)['rougeL'].fmeasure, 4)


# ── Compute scores ─────────────────────────────────────────────────────────
score_rows, avg_rows = [], []

# Zero-shot baseline (empty output)
baseline_rouges = [rouge_l(s['commentary'], '') for s in test_samples]
baseline_bleus  = [bleu(s['commentary'], '')    for s in test_samples]
avg_rows.append({
    'Modèle':        '— Baseline (vide) —',
    'BLEU moyen':    0.0,
    'ROUGE-L moyen': 0.0,
    'VRAM (GB)':     '—',
    'Temps moy.(s)': '—',
})

for r in results:
    if r['status'] != 'success': continue
    bleus, rouges = [], []
    for o in r['outputs']:
        b  = bleu(o['reference'],    o['output'])
        rl = rouge_l(o['reference'], o['output'])
        bleus.append(b); rouges.append(rl)
        score_rows.append({
            'Modèle':  r['model_id'],
            'Image':   f'Img {o["idx"]} ({o["fiche"]})',
            'Famille': o['family'],
            'BLEU':    b,
            'ROUGE-L': rl,
        })
    avg_rows.append({
        'Modèle':        r['model_id'],
        'BLEU moyen':    round(sum(bleus)  / len(bleus),  4),
        'ROUGE-L moyen': round(sum(rouges) / len(rouges), 4),
        'VRAM (GB)':     r['vram_gb'],
        'Temps moy.(s)': r['avg_time_s'],
    })

df_scores = pd.DataFrame(score_rows)
df_avg    = pd.DataFrame(avg_rows).sort_values('ROUGE-L moyen', ascending=False)

print('── BLEU & ROUGE-L par modèle et par image ───────────────────────────')
display(df_scores.to_string(index=False))
print('\n── Scores moyens sur 10 images (trié par ROUGE-L) ───────────────────')
display(df_avg.to_string(index=False))

df_scores.to_csv(os.path.join(RESULTS_DIR, 'bleu_rouge_scores_v4.csv'),   index=False)
df_avg.to_csv(   os.path.join(RESULTS_DIR, 'bleu_rouge_avg_v4.csv'),      index=False)
print(f'\n✓ CSV sauvegardés dans {RESULTS_DIR}')

---
## 6 — Analyse qualitative : structure ITSEOA

Vérifie si les modèles produisent spontanément les 3 composantes du commentaire ITSEOA
(Causes / Gravité / Suites à donner) **sans fine-tuning**.

⚠ Ce n'est pas un critère de scoring — c'est une observation qualitative.  
On s'attend à ce qu'aucun modèle ne produise la structure ITSEOA complète zero-shot,
ce qui justifie la nécessité du fine-tuning en Phase 1.

In [ ]:
print('── Détection de la structure ITSEOA (zero-shot) ─────────────────────')
print('   cause=présence de causes probables | gravité=niveau de sévérité | suites=recommandations')
print()

itseoa_rows = []
for r in results:
    if r['status'] != 'success': continue
    cause_hits   = sum(1 for o in r['outputs'] if o.get('itseoa_hits', {}).get('cause',   False))
    gravite_hits = sum(1 for o in r['outputs'] if o.get('itseoa_hits', {}).get('gravité', False))
    suites_hits  = sum(1 for o in r['outputs'] if o.get('itseoa_hits', {}).get('suites',  False))
    n = len(r['outputs'])
    itseoa_rows.append({
        'Modèle':            r['model_id'],
        'Cause (/10)':       cause_hits,
        'Gravité (/10)':     gravite_hits,
        'Suites (/10)':      suites_hits,
        'Score struct. /30': cause_hits + gravite_hits + suites_hits,
    })

df_itseoa = pd.DataFrame(itseoa_rows).sort_values('Score struct. /30', ascending=False)
display(df_itseoa.to_string(index=False))

print()
print('Note : même avec un score struct. élevé, les termes détectés sont des approximations')
print('génériques — pas la terminologie ITSEOA spécifique. Le fine-tuning Phase 1 est nécessaire.')
df_itseoa.to_csv(os.path.join(RESULTS_DIR, 'itseoa_structure_v4.csv'), index=False)

---
## 7 — Affichage des sorties brutes

Comparaison des sorties de chaque modèle sur une image représentative par famille.

In [ ]:
# Show one representative image per family
shown_families = set()
representative = []
for s in test_samples:
    if s['family'] not in shown_families:
        representative.append(s)
        shown_families.add(s['family'])

for s in representative:
    idx = s['idx']
    print(f'\n{"="*70}')
    print(f'IMAGE {idx} | Fiche {s["fiche"]} | {s["family"]}')
    print(f'{s["desc"]}')
    print(f'{"="*70}')
    thumb = s['image'].copy(); thumb.thumbnail((350, 350)); display(thumb)

    print('\n── COMMENTAIRE EXPERT (référence) ──────────────────────────────────')
    print(textwrap.fill(s['commentary'][:500], width=70))
    if len(s['commentary']) > 500: print('[...]')

    for r in sorted(results, key=lambda x: x.get('leaderboard_rank') or 99):
        if r['status'] != 'success': continue
        oe = next((o for o in r['outputs'] if o['idx'] == idx), None)
        if not oe: continue
        rank_str = f'rank #{r["leaderboard_rank"]}' if r.get('leaderboard_rank') else 'addendum'
        rl = rouge_l(oe['reference'], oe['output'])
        print(f'\n── {r["model_id"]} [{rank_str}] · ROUGE-L={rl:.3f} ──────────────────────')
        print(textwrap.fill(oe['output'][:500], width=70))
        if len(oe['output']) > 500: print('[...]')

---
## 8 — Scores de Référence Publiés

In [ ]:
bench = pd.DataFrame({
    'Modèle':             ['InternVL3-2B',  'Qwen2.5-VL-3B', 'Qwen3-VL-2B', 'InternVL3.5-2B'],
    'Params':             ['2B',            '3B',             '2B',           '2.35B'],
    'Licence':            ['MIT',           'Apache 2.0',     'Apache 2.0',   'Apache 2.0'],
    'Rank (leaderboard)': [10,              11,               '—',            '—'],
    'Avg OVLMBoard':      [64.5,            64.5,             '—',            '—'],
    'MMBench V1.1':       [78.0,            76.8,             '—',            '—'],
    'MMStar':             [61.1,            56.3,             '—',            '—'],
    'OCRBench':           [831,             828,              '—',            '—'],
    'LF template':        ['intern_vl',     'qwen2_5_vl',     'qwen3_vl',     'intern_vl'],
    'Source':             ['Leaderboard',   'Leaderboard',    'Addendum',     'Addendum'],
})
print('── Scores publiés (Open VLM Leaderboard + rapports officiels, Avril 2026) ──')
display(bench.to_string(index=False))
bench.to_csv(os.path.join(RESULTS_DIR, 'published_benchmarks_v4.csv'), index=False)

---
## 9 — Décision Finale

| Critère | Poids | Justification |
|---------|-------|---------------|
| ROUGE-L moyen (10 images ITSEOA) | 35% | Mesure directe sur notre tâche |
| Hardware (VRAM + vitesse) | 25% | Critique pour fine-tuning T4 |
| Langue française | 20% | Données d'entraînement publiées |
| Fine-tuning LLaMA-Factory | 20% | Template confirmé manuellement |

Note : le score leaderboard public (25% dans v3) a été retiré car il favorisait
Qwen2.5-VL-3B sur des benchmarks génériques non représentatifs de notre tâche ITSEOA.

In [ ]:
def score_rouge_criterion(model_id):
    for r in results:
        if r['model_id'] == model_id and r['status'] == 'success':
            rls = [rouge_l(o['reference'], o['output']) for o in r['outputs']]
            return round((sum(rls) / len(rls)) / 0.30 * 35, 1)
    return 0.0


def score_french(model_id):
    if 'Qwen'     in model_id: return 18.0
    if 'InternVL' in model_id: return 15.0
    return 8.0


def score_ft(model_id):
    return 20.0  # All 4 models have confirmed LF templates


def score_hardware(model_id):
    for r in results:
        if r['model_id'] == model_id and r['status'] == 'success':
            vram = r['vram_gb'] or 15
            spd  = r['avg_tok_per_s'] or 1
            vram_score  = max(0, (5.0 - vram) / 5.0 * 12.5)
            speed_score = min(spd / 8.0, 1.0) * 12.5
            return round(vram_score + speed_score, 1)
    return 0.0


rows_final = []
for _, row in bench.iterrows():
    mid = row['Modèle']
    rs  = score_rouge_criterion(mid)
    fs  = score_french(mid)
    fts = score_ft(mid)
    hs  = score_hardware(mid)
    rows_final.append({
        'Modèle':          mid,
        'Rank':            row['Rank (leaderboard)'],
        'ROUGE-L /35':     rs,
        'Matériel /25':    hs,
        'Français /20':    fs,
        'Fine-tuning /20': fts,
        'TOTAL /100':      round(rs + hs + fs + fts, 1),
    })

df_final     = pd.DataFrame(rows_final).sort_values('TOTAL /100', ascending=False)
winner       = df_final.iloc[0]['Modèle']
winner_total = df_final.iloc[0]['TOTAL /100']
winner_rank  = df_final.iloc[0]['Rank']
winner_tpl   = next(m['lf_tpl'] for m in MODEL_REGISTRY if m['id'] == winner)

print('── Scores pondérés finaux ───────────────────────────────────────────')
display(df_final.to_string(index=False))

print(f'\n{"="*60}')
print(f'  MODÈLE RETENU  : {winner}')
print(f'  Leaderboard    : #{winner_rank}')
print(f'  Score pondéré  : {winner_total}/100')
print(f'  LF template    : {winner_tpl}')
print(f'{"="*60}')

decision = {
    'selected_model':       winner,
    'leaderboard_rank':     str(winner_rank),
    'weighted_score':       float(winner_total),
    'lf_template':          winner_tpl,
    'scores_detail':        df_final.to_dict('records'),
    'scoring_weights': {
        'rouge_l':      '35% — mesure directe sur 10 images ITSEOA',
        'hardware':     '25% — VRAM + vitesse, critique pour fine-tuning T4',
        'french':       '20% — données d\'entraînement publiées',
        'finetuning':   '20% — template LLaMA-Factory confirmé manuellement',
        'leaderboard':  '0%  — retiré (benchmarks génériques non représentatifs)',
    },
    'methodology': (
        'Phase initiale : Open VLM Leaderboard top 15 (<=4B), filtré par '
        'disponibilité HF, interface standard, support LLaMA-Factory, licence Apache 2.0 ou MIT. '
        'Addendum : révision systématique LLaMA-Factory vs sorties HuggingFace Mars-Avril 2026. '
        'Évaluation zero-shot sur 10 images ITSEOA stratifiées par famille de désordre.'
    ),
    'excluded_models': [
        {'id': 'SmolVLM2-2.2B',   'reason': 'Pas de template LLaMA-Factory'},
        {'id': 'Gemma3-4B',       'reason': '4B à la limite VRAM T4 ; licence Gemma Terms of Use'},
        {'id': 'Qwen2.5-Omni-3B', 'reason': 'Licence qwen-research (recherche uniquement)'},
        {'id': 'Qwen3.5-VL',      'reason': "N'existe pas : Qwen3.5 est texte uniquement"},
        {'id': 'LFM2.5-VL',       'reason': 'Licence LFM 1.0 (restreint < 10M$ CA)'},
    ],
    'itseoa_note': (
        'Aucun modèle ne produit la structure ITSEOA complète (Causes/Gravité/Suites) '
        'zero-shot. Les occurrences détectées sont des termes génériques, pas la '
        'terminologie ITSEOA spécifique. Ceci confirme la nécessité du fine-tuning Phase 1.'
    ),
    'experimental_results': results,
}

out_path = os.path.join(RESULTS_DIR, 'final_decision_v4.json')
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(decision, f, ensure_ascii=False, indent=2)
df_final.to_csv(os.path.join(RESULTS_DIR, 'model_weighted_scores_v4.csv'), index=False)

print(f'\n✓ Décision sauvegardée : {out_path}')
print(f'Prochain jalon : Phase 1 — Fine-tuning QLoRA avec LLaMA-Factory (template: {winner_tpl})')

---
## Résumé / Summary

**Fichiers produits dans `{CID_BASE}/phase0/` :**
- `model_selection_results_v4.json` — résultats expérimentaux complets (10 images)
- `bleu_rouge_scores_v4.csv` — scores BLEU/ROUGE-L par modèle et par image
- `bleu_rouge_avg_v4.csv` — moyennes par modèle + baseline zero-shot
- `itseoa_structure_v4.csv` — détection structure Causes/Gravité/Suites
- `published_benchmarks_v4.csv` — scores de référence publiés
- `model_weighted_scores_v4.csv` — scores pondérés finaux
- `final_decision_v4.json` — décision documentée complète

*Ces résultats alimentent le rapport de Phase 0 et justifient le choix du modèle pour Phase 1.*